# 03 — Random Forest feature check

기존 02 notebook이 만든 train engineered-feature CSV만 사용합니다. preprocessing, windowing, scaling을 다시 하지 않습니다. 각 CSV의 실제 header에서 104개 feature와 세 TTF column을 확인합니다. 공식 test dataset은 읽지 않습니다.

100/50 overlapping windows의 누수를 막기 위해 모든 source_file + sequence_index group을 먼저 GroupShuffleSplit으로 나눕니다. 각 target의 전체 usable row 수를 표시하고, 계산 시간을 제한하기 위해 RF에는 각 split에서 target별 최대 100,000 train / 25,000 validation row를 group마다 고르게 뽑아 사용합니다. 표본은 재현 가능하며, full group split과 disjoint assertion은 그대로 유지됩니다.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from joblib import parallel_backend

SEED = 42
SCAN_CHUNK_ROWS = 100_000
SAMPLE_CHUNK_ROWS = 50_000
MAX_TRAIN_ROWS_PER_TARGET = 100_000
MAX_VALIDATION_ROWS_PER_TARGET = 25_000
N_ESTIMATORS = 200
MIN_SAMPLES_LEAF = 5
PERMUTATION_REPEATS = 5
TOP_N = 20
CORRELATION_THRESHOLD = 0.95


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "interim" / "phm2018_jev" / "features").is_dir():
            return candidate
    raise FileNotFoundError("Could not find existing PHM2018 engineered features.")


PROJECT_ROOT = find_project_root()
FEATURE_TRAIN_DIR = PROJECT_ROOT / "data" / "interim" / "phm2018_jev" / "features" / "train"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "random_forest"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("scikit-learn:", sklearn.__version__)
print("Train feature directory:", FEATURE_TRAIN_DIR)
print("Output directory:", ARTIFACT_DIR)

# Check native missing-value support in the installed RandomForestRegressor.
_nan_probe = RandomForestRegressor(n_estimators=5, random_state=SEED, n_jobs=1)
try:
    _nan_probe.fit(np.array([[0.0], [np.nan], [1.0], [2.0]]), np.array([0.0, 1.0, 2.0, 3.0]))
    _nan_probe.predict(np.array([[np.nan], [0.5]]))
    RF_SUPPORTS_NAN = True
except (TypeError, ValueError):
    RF_SUPPORTS_NAN = False
print("RandomForestRegressor native NaN support:", RF_SUPPORTS_NAN)

## 1. Inspect existing feature files

이 셀은 train feature 파일의 header와 일부가 아닌 전체 metadata/target 열을 chunk 단위로 확인합니다. feature 계산에는 기존 산출물만 사용하고 test feature 폴더는 열지 않습니다.

In [ ]:
train_paths = sorted(FEATURE_TRAIN_DIR.glob("*.csv"))
if not train_paths:
    raise FileNotFoundError(f"No existing train feature CSVs found in {FEATURE_TRAIN_DIR}")

headers_by_file = {path.name: pd.read_csv(path, nrows=0).columns.tolist() for path in train_paths}
header = headers_by_file[train_paths[0].name]
for name, columns in headers_by_file.items():
    if columns != header:
        raise ValueError(f"CSV header differs from the first train file: {name}")

STATISTICS = {
    "mean", "std", "peak_abs", "rms", "kurtosis_excess",
    "skewness", "crest_factor", "shape_factor",
}
FEATURE_COLUMNS = [
    column for column in header
    if "__" in column and column.rsplit("__", 1)[1] in STATISTICS
]
TARGET_COLUMNS = [
    column for column in header
    if column.startswith("ttf_") and column.endswith("_seconds")
]
REQUIRED_COLUMNS = {"source_file", "sequence_index", "Tool", "window_index"}
missing_required = REQUIRED_COLUMNS.difference(header)
if missing_required:
    raise ValueError(f"Required metadata is absent from the actual CSV header: {sorted(missing_required)}")
if len(FEATURE_COLUMNS) != 104:
    raise ValueError(f"Expected 104 existing engineered features; found {len(FEATURE_COLUMNS)}")
if len(TARGET_COLUMNS) != 3:
    raise ValueError(f"Expected three existing TTF target columns; found {TARGET_COLUMNS}")

signature_payload = {
    "train_files": [(path.name, path.stat().st_size, int(path.stat().st_mtime_ns)) for path in train_paths],
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "seed": SEED,
    "validation_fraction": 0.20,
    "n_estimators": N_ESTIMATORS,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "max_train_rows_per_target": MAX_TRAIN_ROWS_PER_TARGET,
    "max_validation_rows_per_target": MAX_VALIDATION_ROWS_PER_TARGET,
    "permutation_repeats": PERMUTATION_REPEATS,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode("utf-8")
).hexdigest()
CACHE_RESULT_PATHS = [
    ARTIFACT_DIR / "rf_metrics.csv",
    ARTIFACT_DIR / "rf_feature_importance.csv",
    ARTIFACT_DIR / "permutation_importance.csv",
]
def cache_file_matches(path):
    if not path.is_file():
        return False
    columns = pd.read_csv(path, nrows=0).columns
    if "analysis_signature" not in columns:
        return False
    values = pd.read_csv(path, usecols=["analysis_signature"], nrows=1)["analysis_signature"]
    return len(values) == 1 and values.iloc[0] == RUN_SIGNATURE
CACHED_RF_RESULTS = all(cache_file_matches(path) for path in CACHE_RESULT_PATHS)
print("Matching saved RF results:", CACHED_RF_RESULTS)

for column in FEATURE_COLUMNS:
    if column in TARGET_COLUMNS or column in REQUIRED_COLUMNS:
        raise AssertionError(f"Non-feature column entered the model feature list: {column}")

print(f"Train CSV files: {len(train_paths)}")
print(f"Train CSV size: {sum(path.stat().st_size for path in train_paths) / (1024 ** 3):.2f} GiB")
print(f"Existing feature columns: {len(FEATURE_COLUMNS)}")
print("Actual feature columns:", FEATURE_COLUMNS)
print("Actual TTF columns:", TARGET_COLUMNS)

# One pass over only identifiers and target columns builds the complete grouped split inventory.
scan_columns = ["source_file", "sequence_index", "Tool", *TARGET_COLUMNS]
group_records = {}
total_window_rows = 0
for file_number, path in enumerate(train_paths, start=1):
    print(f"Scanning group/target counts {file_number}/{len(train_paths)}: {path.name}")
    for chunk in pd.read_csv(
        path, usecols=scan_columns, chunksize=SCAN_CHUNK_ROWS, low_memory=False
    ):
        total_window_rows += len(chunk)
        if chunk["source_file"].isna().any() or chunk["sequence_index"].isna().any():
            raise ValueError(f"Missing source_file or sequence_index in {path.name}")
        chunk["group_id"] = (
            chunk["source_file"].astype(str) + "::" + chunk["sequence_index"].astype(str)
        )
        agg_spec = {
            "source_file": ("source_file", "first"),
            "Tool": ("Tool", "first"),
            "total_windows": ("group_id", "size"),
        }
        agg_spec.update({target: (target, "count") for target in TARGET_COLUMNS})
        chunk_groups = chunk.groupby("group_id", sort=False, dropna=False).agg(**agg_spec)
        for group_id, row in chunk_groups.iterrows():
            if group_id not in group_records:
                group_records[group_id] = {
                    "group_id": group_id,
                    "source_file": str(row["source_file"]),
                    "Tool": str(row["Tool"]),
                    "total_windows": int(row["total_windows"]),
                    **{target: int(row[target]) for target in TARGET_COLUMNS},
                }
            else:
                record = group_records[group_id]
                record["total_windows"] += int(row["total_windows"])
                for target in TARGET_COLUMNS:
                    record[target] += int(row[target])

group_table = pd.DataFrame(group_records.values()).sort_values("group_id").reset_index(drop=True)
if group_table["group_id"].duplicated().any():
    raise AssertionError("Group inventory must contain one row per source_file + sequence_index.")

target_labels = {
    "ttf_flowcool_pressure_low_seconds": "FlowCool pressure low",
    "ttf_flowcool_pressure_high_seconds": "FlowCool pressure high",
    "ttf_flowcool_leak_seconds": "FlowCool leak",
}
def target_label(target):
    return target_labels.get(target, target.removeprefix("ttf_").removesuffix("_seconds").replace("_", " "))

target_usable = pd.DataFrame([
    {
        "target": target_label(target),
        "target_column": target,
        "usable_samples": int(group_table[target].sum()),
        "nan_samples": int(total_window_rows - group_table[target].sum()),
        "nan_fraction": float(1 - group_table[target].sum() / total_window_rows),
    }
    for target in TARGET_COLUMNS
])
print(f"Total train windows: {total_window_rows:,}")
print(f"Source-file + sequence groups: {len(group_table):,}")
display(target_usable[["target", "usable_samples"]])


## 2. Grouped train/validation split

모든 group을 한 번만 80/20으로 나누며, split은 target별 샘플 추출 전에 고정합니다. target이 null인 행은 해당 target 모델에서만 제외됩니다.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_group_idx, validation_group_idx = next(
    splitter.split(group_table, groups=group_table["group_id"])
)
train_groups = set(group_table.iloc[train_group_idx]["group_id"])
validation_groups = set(group_table.iloc[validation_group_idx]["group_id"])

assert train_groups.isdisjoint(validation_groups), "Train/validation group leakage detected."
assert train_groups | validation_groups == set(group_table["group_id"]), "Some groups were not assigned."

group_table["split"] = np.where(
    group_table["group_id"].isin(train_groups), "train", "validation"
)
group_summary = (
    group_table.groupby("split")
    .agg(
        groups=("group_id", "nunique"),
        source_files=("source_file", "nunique"),
        tools=("Tool", "nunique"),
        windows=("total_windows", "sum"),
    )
    .reindex(["train", "validation"])
    .reset_index()
)
split_target_rows = []
for target in TARGET_COLUMNS:
    for split_name in ("train", "validation"):
        rows = group_table.loc[group_table["split"].eq(split_name), target].sum()
        split_target_rows.append({
            "target": target_label(target),
            "split": split_name,
            "usable_samples": int(rows),
        })
target_split_counts = pd.DataFrame(split_target_rows)

print(f"Train groups: {len(train_groups):,}")
print(f"Validation groups: {len(validation_groups):,}")
print(f"Group overlap: {len(train_groups & validation_groups)}")
display(group_summary)
display(target_split_counts.pivot(index="target", columns="split", values="usable_samples"))

# Save the exact fixed split and group-level target counts for downstream TTF baselines.
SPLIT_ARTIFACT_PATH = ARTIFACT_DIR / "ttf_baseline_group_split.csv"
group_table.to_csv(SPLIT_ARTIFACT_PATH, index=False)
SPLIT_METADATA_PATH = ARTIFACT_DIR / "ttf_baseline_split_metadata.json"
with SPLIT_METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "random_seed": SEED,
            "validation_fraction": 0.20,
            "group_definition": "source_file + '::' + sequence_index",
            "target_columns": TARGET_COLUMNS,
            "target_labels": target_labels,
            "train_groups": len(train_groups),
            "validation_groups": len(validation_groups),
            "group_overlap": len(train_groups & validation_groups),
        },
        f,
        indent=2,
    )
print("Saved fixed grouped split:", SPLIT_ARTIFACT_PATH)


## 3. Deterministic, group-balanced RF sample

학습용 표본은 각 target/split의 non-null 행에서 group별로 균등하게 quota를 배분하고, sequence 순서상 고르게 선택합니다. 총 표본은 target별 train 최대 100,000행, validation 최대 25,000행입니다. group 분할은 표본 크기와 무관하게 앞 셀의 전체 group inventory에서 완료되었습니다.

같은 train 파일 상태와 분석 설정으로 만든 RF 결과 CSV가 있으면 서명을 확인해 재사용합니다. 파일이나 설정이 달라지면 표본을 다시 만들고 모델을 학습합니다.

In [ ]:
if CACHED_RF_RESULTS:
    cached_metrics = pd.read_csv(ARTIFACT_DIR / "rf_metrics.csv")
    sample_rows = []
    for row in cached_metrics.itertuples(index=False):
        sample_rows.append({
            "target": row.target, "split": "train",
            "full_usable_samples": row.full_train_samples,
            "rf_sample_rows": row.rf_train_rows,
            "sampled_groups": "cached result",
        })
        sample_rows.append({
            "target": row.target, "split": "validation",
            "full_usable_samples": row.full_validation_samples,
            "rf_sample_rows": row.rf_validation_rows,
            "sampled_groups": "cached result",
        })
    sample_summary = pd.DataFrame(sample_rows)
    print("Using matching saved RF sample counts; feature CSVs are not reread for model fitting.")
    display(sample_summary)
else:
    def allocate_quotas(group_counts, cap):
        counts = {group_id: int(count) for group_id, count in group_counts.items() if int(count) > 0}
        total = sum(counts.values())
        budget = min(int(cap), total)
        quotas = {group_id: 0 for group_id in counts}
        active = list(counts)
        remaining = budget

        while active and remaining > 0:
            per_group, extra = divmod(remaining, len(active))
            if per_group == 0:
                for group_id in active[:remaining]:
                    quotas[group_id] += 1
                break
            saturated = [
                group_id for group_id in active
                if counts[group_id] - quotas[group_id] <= per_group
            ]
            if saturated:
                for group_id in saturated:
                    available = counts[group_id] - quotas[group_id]
                    quotas[group_id] += available
                    remaining -= available
                active = [group_id for group_id in active if group_id not in set(saturated)]
            else:
                for position, group_id in enumerate(active):
                    quotas[group_id] += per_group + int(position < extra)
                remaining = 0
        return {group_id: quota for group_id, quota in quotas.items() if quota > 0}


    split_limits = {
        "train": MAX_TRAIN_ROWS_PER_TARGET,
        "validation": MAX_VALIDATION_ROWS_PER_TARGET,
    }
    sample_plan = {}
    sample_cursors = {}
    quota_rows = []
    for target in TARGET_COLUMNS:
        for split_name, split_group_set in [
            ("train", train_groups), ("validation", validation_groups)
        ]:
            counts = group_table.loc[
                group_table["group_id"].isin(split_group_set), ["group_id", target]
            ].set_index("group_id")[target]
            quotas = allocate_quotas(counts.to_dict(), split_limits[split_name])
            selections = {
                group_id: np.linspace(0, int(counts[group_id]) - 1, num=quota, dtype=np.int64)
                for group_id, quota in quotas.items()
            }
            key = (target, split_name)
            sample_plan[key] = selections
            sample_cursors[key] = {}
            quota_rows.append({
                "target": target_label(target),
                "split": split_name,
                "full_usable_samples": int(counts.sum()),
                "sample_quota": int(sum(quotas.values())),
                "groups_with_usable_labels": int((counts > 0).sum()),
            })
    sample_quota_table = pd.DataFrame(quota_rows)
    display(sample_quota_table)

    sample_columns = [
        "source_file", "sequence_index", *FEATURE_COLUMNS, *TARGET_COLUMNS
    ]
    sample_parts = {(target, split): [] for target in TARGET_COLUMNS for split in ("train", "validation")}
    split_lookup = {group_id: "train" for group_id in train_groups}
    split_lookup.update({group_id: "validation" for group_id in validation_groups})

    for file_number, path in enumerate(train_paths, start=1):
        print(f"Reading existing features for RF sample {file_number}/{len(train_paths)}: {path.name}")
        for chunk in pd.read_csv(
            path, usecols=sample_columns, chunksize=SAMPLE_CHUNK_ROWS, low_memory=False
        ):
            chunk["group_id"] = (
                chunk["source_file"].astype(str) + "::" + chunk["sequence_index"].astype(str)
            )
            chunk["split"] = chunk["group_id"].map(split_lookup)
            for target in TARGET_COLUMNS:
                valid_target = chunk[target].notna()
                for split_name in ("train", "validation"):
                    candidate_rows = chunk.loc[
                        valid_target & chunk["split"].eq(split_name),
                        ["group_id", *FEATURE_COLUMNS, target],
                    ]
                    if candidate_rows.empty:
                        continue
                    key = (target, split_name)
                    selection_by_group = sample_plan[key]
                    cursor_by_group = sample_cursors[key]
                    for group_id, group_rows in candidate_rows.groupby("group_id", sort=False):
                        start_ordinal = cursor_by_group.get(group_id, 0)
                        row_ordinals = start_ordinal + np.arange(len(group_rows), dtype=np.int64)
                        wanted_ordinals = selection_by_group.get(group_id)
                        if wanted_ordinals is not None:
                            keep = np.isin(row_ordinals, wanted_ordinals, assume_unique=True)
                            if keep.any():
                                sample_parts[key].append(group_rows.iloc[np.flatnonzero(keep)].copy())
                        cursor_by_group[group_id] = start_ordinal + len(group_rows)

    sampled_data = {}
    for target in TARGET_COLUMNS:
        for split_name in ("train", "validation"):
            key = (target, split_name)
            parts = sample_parts[key]
            if parts:
                frame = pd.concat(parts, ignore_index=True)
            else:
                frame = pd.DataFrame(columns=["group_id", *FEATURE_COLUMNS, target])
            expected_rows = sum(len(indices) for indices in sample_plan[key].values())
            if len(frame) != expected_rows:
                raise AssertionError(
                    f"Sample selection mismatch for {target}/{split_name}: "
                    f"expected {expected_rows}, found {len(frame)}"
                )
            sampled_data[key] = frame

    sample_summary_rows = []
    for target in TARGET_COLUMNS:
        for split_name in ("train", "validation"):
            key = (target, split_name)
            sample_summary_rows.append({
                "target": target_label(target),
                "split": split_name,
                "full_usable_samples": int(
                    group_table.loc[group_table["group_id"].isin(
                        train_groups if split_name == "train" else validation_groups
                    ), target].sum()
                ),
                "rf_sample_rows": len(sampled_data[key]),
                "sampled_groups": sampled_data[key]["group_id"].nunique(),
            })
    sample_summary = pd.DataFrame(sample_summary_rows)
    display(sample_summary)


## 4. Baseline metrics and feature rankings

각 target은 별도 RandomForestRegressor로 확인합니다. sklearn이 feature NaN을 직접 처리하면 imputation 없이 사용하고, 지원하지 않는 환경에서만 train sample에 fit된 median imputer를 사용합니다. Permutation importance는 validation R² 기준 5회 반복입니다.

In [ ]:
if CACHED_RF_RESULTS:
    rf_metrics = pd.read_csv(ARTIFACT_DIR / "rf_metrics.csv")
    rf_importance = pd.read_csv(ARTIFACT_DIR / "rf_feature_importance.csv")
    permutation_results = pd.read_csv(ARTIFACT_DIR / "permutation_importance.csv")
    print("Loaded matching saved RF metrics and importance tables.")
    display(rf_metrics[
        [
            "target", "train_MAE", "validation_MAE", "train_RMSE", "validation_RMSE",
            "train_R2", "validation_R2", "rf_train_rows", "rf_validation_rows", "overfit_flag",
        ]
    ])
    for target in TARGET_COLUMNS:
        target_name = target_label(target)
        print(f"RF impurity top 20 — {target_name}")
        display(
            rf_importance.loc[rf_importance["target"].eq(target_name)]
            .sort_values("rank").head(TOP_N)
        )
        print(f"Permutation top 20 — {target_name}")
        display(
            permutation_results.loc[permutation_results["target"].eq(target_name)]
            .sort_values("rank").head(TOP_N)
        )
else:
    metric_rows = []
    rf_importance_frames = []
    permutation_frames = []
    fitted_models = {}
    fit_seconds = {}

    def regression_metrics(y_true, y_pred):
        return {
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2": float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan,
        }

    for target in TARGET_COLUMNS:
        train_frame = sampled_data[(target, "train")]
        validation_frame = sampled_data[(target, "validation")]
        if len(train_frame) < 2 or len(validation_frame) < 2:
            metric_rows.append({
                "target": target_label(target),
                "target_column": target,
                "train_MAE": np.nan, "validation_MAE": np.nan,
                "train_RMSE": np.nan, "validation_RMSE": np.nan,
                "train_R2": np.nan, "validation_R2": np.nan,
                "full_train_samples": int(group_table.loc[group_table["group_id"].isin(train_groups), target].sum()),
                "full_validation_samples": int(group_table.loc[group_table["group_id"].isin(validation_groups), target].sum()),
                "rf_train_rows": len(train_frame), "rf_validation_rows": len(validation_frame),
                "fit_seconds": np.nan, "overfit_flag": "insufficient grouped samples",
            })
            print(f"Skipping {target}: too few sampled rows for grouped regression.")
            continue

        y_train = pd.to_numeric(train_frame[target], errors="raise").to_numpy(dtype=np.float64)
        y_validation = pd.to_numeric(validation_frame[target], errors="raise").to_numpy(dtype=np.float64)
        if not np.isfinite(y_train).all() or not np.isfinite(y_validation).all():
            raise ValueError(f"Non-finite non-null target values found in {target}")

        X_train_full = train_frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
        X_validation_full = validation_frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
        active_features = [column for column in FEATURE_COLUMNS if X_train_full[column].notna().any()]
        all_missing_features = sorted(set(FEATURE_COLUMNS) - set(active_features))
        if not active_features:
            raise ValueError(f"All 104 features are missing in the train sample for {target}")
        X_train = X_train_full[active_features]
        X_validation = X_validation_full[active_features]

        rf = RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            random_state=SEED,
            n_jobs=-1,
        )
        if RF_SUPPORTS_NAN:
            estimator = rf
            nan_method = "native RandomForestRegressor NaN handling"
        else:
            estimator = make_pipeline(SimpleImputer(strategy="median"), rf)
            nan_method = "training-sample median imputer"

        fit_started = time.perf_counter()
        estimator.fit(X_train, y_train)
        elapsed = time.perf_counter() - fit_started
        fitted_models[target] = estimator
        fit_seconds[target] = elapsed
        rf_model = estimator.named_steps["randomforestregressor"] if not RF_SUPPORTS_NAN else estimator

        train_pred = estimator.predict(X_train)
        validation_pred = estimator.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_pred)
        validation_metrics = regression_metrics(y_validation, validation_pred)

        if validation_metrics["MAE"] > 1.5 * max(train_metrics["MAE"], 1e-12) or (
            np.isfinite(train_metrics["R2"]) and np.isfinite(validation_metrics["R2"])
            and train_metrics["R2"] - validation_metrics["R2"] > 0.25
        ):
            overfit_flag = "possible overfit: train/validation gap is large"
        else:
            overfit_flag = "no large train/validation gap by this simple check"

        full_train_samples = int(group_table.loc[group_table["group_id"].isin(train_groups), target].sum())
        full_validation_samples = int(group_table.loc[group_table["group_id"].isin(validation_groups), target].sum())
        metric_rows.append({
            "target": target_label(target),
            "target_column": target,
            "train_MAE": train_metrics["MAE"],
            "validation_MAE": validation_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "validation_RMSE": validation_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "validation_R2": validation_metrics["R2"],
            "full_train_samples": full_train_samples,
            "full_validation_samples": full_validation_samples,
            "rf_train_rows": len(train_frame),
            "rf_validation_rows": len(validation_frame),
            "fit_seconds": elapsed,
            "overfit_flag": overfit_flag,
        })
        print(
            f"{target_label(target)}: fit {elapsed:.1f}s; {nan_method}; "
            f"train rows={len(train_frame):,}, validation rows={len(validation_frame):,}; "
            f"validation R2={validation_metrics['R2']:.4f}"
        )

        rf_values = dict(zip(active_features, rf_model.feature_importances_))
        rf_frame = pd.DataFrame({"feature": FEATURE_COLUMNS})
        rf_frame["rf_importance"] = rf_frame["feature"].map(rf_values).fillna(0.0)
        rf_frame["target"] = target_label(target)
        rf_frame["target_column"] = target
        rf_frame["sensor"] = rf_frame["feature"].str.rsplit("__", n=1).str[0]
        rf_frame["statistic"] = rf_frame["feature"].str.rsplit("__", n=1).str[1]
        rf_frame["rank"] = rf_frame["rf_importance"].rank(method="min", ascending=False).astype(int)
        rf_importance_frames.append(rf_frame[
            ["target", "target_column", "rank", "feature", "sensor", "statistic", "rf_importance"]
        ])

        with parallel_backend("threading"):
            perm_result = permutation_importance(
                estimator,
                X_validation,
                y_validation,
                scoring="r2",
                n_repeats=PERMUTATION_REPEATS,
                random_state=SEED,
                n_jobs=-1,
            )
        perm_values = dict(zip(active_features, perm_result.importances_mean))
        perm_stds = dict(zip(active_features, perm_result.importances_std))
        perm_frame = pd.DataFrame({"feature": FEATURE_COLUMNS})
        perm_frame["permutation_importance_mean"] = perm_frame["feature"].map(perm_values).fillna(0.0)
        perm_frame["permutation_importance_std"] = perm_frame["feature"].map(perm_stds).fillna(0.0)
        perm_frame["target"] = target_label(target)
        perm_frame["target_column"] = target
        perm_frame["sensor"] = perm_frame["feature"].str.rsplit("__", n=1).str[0]
        perm_frame["statistic"] = perm_frame["feature"].str.rsplit("__", n=1).str[1]
        perm_frame["rank"] = perm_frame["permutation_importance_mean"].rank(
            method="min", ascending=False
        ).astype(int)
        perm_frame["n_repeats"] = PERMUTATION_REPEATS
        perm_frame["n_validation_samples"] = len(validation_frame)
        permutation_frames.append(perm_frame[
            [
                "target", "target_column", "rank", "feature", "sensor", "statistic",
                "permutation_importance_mean", "permutation_importance_std",
                "n_repeats", "n_validation_samples",
            ]
        ])

    rf_metrics = pd.DataFrame(metric_rows)
    rf_importance = pd.concat(rf_importance_frames, ignore_index=True) if rf_importance_frames else pd.DataFrame()
    permutation_results = (
        pd.concat(permutation_frames, ignore_index=True) if permutation_frames else pd.DataFrame()
    )
    display(rf_metrics[
        [
            "target", "train_MAE", "validation_MAE", "train_RMSE", "validation_RMSE",
            "train_R2", "validation_R2", "rf_train_rows", "rf_validation_rows", "overfit_flag",
        ]
    ])

    for result_frame in (rf_metrics, rf_importance, permutation_results):
        result_frame["analysis_signature"] = RUN_SIGNATURE
    if not rf_metrics.empty:
        rf_metrics.to_csv(ARTIFACT_DIR / "rf_metrics.csv", index=False)
    if not rf_importance.empty:
        rf_importance.to_csv(ARTIFACT_DIR / "rf_feature_importance.csv", index=False)
    if not permutation_results.empty:
        permutation_results.to_csv(ARTIFACT_DIR / "permutation_importance.csv", index=False)

    for target in TARGET_COLUMNS:
        target_name = target_label(target)
        print(f"RF impurity top 20 — {target_name}")
        display(
            rf_importance.loc[rf_importance["target"].eq(target_name)]
            .sort_values("rank")
            .head(TOP_N)
        )
        print(f"Permutation top 20 — {target_name}")
        display(
            permutation_results.loc[permutation_results["target"].eq(target_name)]
            .sort_values("rank")
            .head(TOP_N)
        )


## 5. Cross-target ranking and redundancy

Permutation ranking을 우선해 target마다 104개 feature 순위를 합칩니다. 상위 30개 feature에서 Pearson correlation 절댓값 0.95 이상인 pair를 표시하며, 자동 삭제하지 않습니다.

In [ ]:
if permutation_results.empty:
    raise RuntimeError("No target model produced permutation importance.")

permutation_rank_wide = permutation_results.pivot(
    index="feature", columns="target_column", values="rank"
).reindex(FEATURE_COLUMNS)
permutation_importance_wide = permutation_results.pivot(
    index="feature", columns="target_column", values="permutation_importance_mean"
).reindex(FEATURE_COLUMNS)

aggregate = pd.DataFrame({"feature": FEATURE_COLUMNS})
aggregate["sensor"] = aggregate["feature"].str.rsplit("__", n=1).str[0]
aggregate["statistic"] = aggregate["feature"].str.rsplit("__", n=1).str[1]
aggregate["target_top20_count"] = (
    permutation_rank_wide[TARGET_COLUMNS].le(TOP_N).sum(axis=1)
).reindex(aggregate["feature"]).to_numpy()
aggregate["mean_rank"] = permutation_rank_wide.mean(axis=1).to_numpy()
aggregate["mean_permutation_importance"] = permutation_importance_wide.mean(axis=1).to_numpy()

for target in TARGET_COLUMNS:
    aggregate[f"rank_{target}"] = permutation_rank_wide[target].to_numpy()
    aggregate[f"permutation_{target}"] = permutation_importance_wide[target].to_numpy()

aggregate = aggregate.sort_values(
    ["target_top20_count", "mean_rank", "mean_permutation_importance"],
    ascending=[False, True, False],
).reset_index(drop=True)
aggregate["aggregate_rank"] = np.arange(1, len(aggregate) + 1)
aggregate.to_csv(ARTIFACT_DIR / "aggregate_feature_ranking.csv", index=False)
display(aggregate.head(20))

# Use the largest target-specific RF train sample for a train-only correlation view.
top30_features = aggregate.head(30)["feature"].tolist()
if "sampled_data" in globals() and all((target, "train") in sampled_data for target in TARGET_COLUMNS):
    corr_target = max(TARGET_COLUMNS, key=lambda target: len(sampled_data[(target, "train")]))
    corr_frame = sampled_data[(corr_target, "train")]
    corr_X = corr_frame[top30_features].replace([np.inf, -np.inf], np.nan)
else:
    corr_parts = [
        pd.read_csv(path, usecols=top30_features, nrows=3_000, low_memory=False)
        for path in train_paths
    ]
    corr_X = pd.concat(corr_parts, ignore_index=True).replace([np.inf, -np.inf], np.nan)
correlation_matrix = corr_X.corr(method="pearson", min_periods=3)

correlated_pairs = []
for i, feature_a in enumerate(top30_features):
    for feature_b in top30_features[i + 1:]:
        value = correlation_matrix.loc[feature_a, feature_b]
        if pd.notna(value) and abs(value) >= CORRELATION_THRESHOLD:
            rank_a = int(aggregate.loc[aggregate["feature"].eq(feature_a), "aggregate_rank"].iloc[0])
            rank_b = int(aggregate.loc[aggregate["feature"].eq(feature_b), "aggregate_rank"].iloc[0])
            representative = feature_a if rank_a < rank_b else feature_b
            correlated_pairs.append({
                "feature_a": feature_a,
                "feature_b": feature_b,
                "correlation": float(value),
                "absolute_correlation": float(abs(value)),
                "representative_candidate": representative,
            })
correlated_pairs = pd.DataFrame(correlated_pairs)
if correlated_pairs.empty:
    correlated_pairs = pd.DataFrame(
        columns=["feature_a", "feature_b", "correlation", "absolute_correlation", "representative_candidate"]
    )
print(f"Top-30 feature pairs with |correlation| >= {CORRELATION_THRESHOLD:.2f}: {len(correlated_pairs)}")
display(correlated_pairs)

# Greedy compact set: first prefer one feature per sensor, then allow up to three.
selected_features = []
sensor_counts = {}
redundancy_notes = {}
top30_ranked = aggregate.head(30)["feature"].tolist()

def correlated_with_selected(feature):
    related = []
    for chosen in selected_features:
        value = correlation_matrix.loc[feature, chosen]
        if pd.notna(value) and abs(value) >= CORRELATION_THRESHOLD:
            related.append((chosen, float(value)))
    return related

for sensor_cap in (1, 3):
    for feature in top30_ranked:
        if len(selected_features) >= 20:
            break
        if feature in selected_features:
            continue
        sensor = aggregate.loc[aggregate["feature"].eq(feature), "sensor"].iloc[0]
        if sensor_counts.get(sensor, 0) >= sensor_cap:
            continue
        redundant = correlated_with_selected(feature)
        if redundant:
            best_existing, corr_value = redundant[0]
            redundancy_notes[feature] = (
                f"Not selected: |r|={abs(corr_value):.3f} with {best_existing}, "
                "which has a stronger aggregate rank."
            )
            continue
        selected_features.append(feature)
        sensor_counts[sensor] = sensor_counts.get(sensor, 0) + 1

candidate_rows = []
for feature in selected_features:
    row = aggregate.loc[aggregate["feature"].eq(feature)].iloc[0]
    related_omitted = []
    for omitted in top30_ranked:
        if omitted == feature or omitted in selected_features:
            continue
        value = correlation_matrix.loc[feature, omitted]
        if pd.notna(value) and abs(value) >= CORRELATION_THRESHOLD:
            related_omitted.append(f"{omitted} (r={value:.2f})")
    coverage = int(row["target_top20_count"])
    candidate_rows.append({
        "feature": feature,
        "sensor": row["sensor"],
        "statistic": row["statistic"],
        "importance_reason": (
            f"Permutation aggregate rank {int(row['aggregate_rank'])}; "
            f"Top 20 in {coverage}/3 targets; mean rank {row['mean_rank']:.1f}."
        ),
        "redundancy_note": (
            "Selected as the higher-ranked representative; overlaps with "
            + ", ".join(related_omitted)
            if related_omitted
            else "No |r| >= 0.95 pair among the top-30 pool."
        ),
    })
jev_candidates = pd.DataFrame(candidate_rows)
jev_candidates.to_csv(ARTIFACT_DIR / "jev_candidate_features.csv", index=False)
print(f"JEV candidate features: {len(jev_candidates)}")
display(jev_candidates)

print("Saved requested result files:")
for name in [
    "rf_metrics.csv",
    "rf_feature_importance.csv",
    "permutation_importance.csv",
    "aggregate_feature_ranking.csv",
    "jev_candidate_features.csv",
]:
    print(" -", ARTIFACT_DIR / name)


## 6. Minimal portfolio figures

아래는 validation 성능, target별 permutation Top 15, 통합 ranking Top 20, 통합 상위 30 feature의 correlation만 보여줍니다.

In [ ]:
# Validation R2 comparison.
plot_metrics = rf_metrics.dropna(subset=["validation_R2"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(plot_metrics["target"], plot_metrics["validation_R2"], color="#376996")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Validation R²")
ax.set_title("Grouped validation R² by TTF target")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
plt.show()

# Target-specific permutation importance Top 15.
fig, axes = plt.subplots(1, len(TARGET_COLUMNS), figsize=(17, 7), squeeze=False)
for axis, target in zip(axes[0], TARGET_COLUMNS):
    target_name = target_label(target)
    top = (
        permutation_results.loc[permutation_results["target"].eq(target_name)]
        .sort_values("rank")
        .head(15)
        .sort_values("permutation_importance_mean")
    )
    axis.barh(top["feature"], top["permutation_importance_mean"], color="#5b8e7d")
    axis.set_title(target_name)
    axis.set_xlabel("R² score decrease")
    axis.tick_params(axis="y", labelsize=7)
fig.suptitle("Permutation importance — validation Top 15")
fig.tight_layout()
plt.show()

# Cross-target aggregate ranking Top 20.
top20_aggregate = aggregate.head(20).sort_values("mean_rank", ascending=False)
fig, ax = plt.subplots(figsize=(10, 7))
colors = [
    "#d08c60" if count == 3 else "#376996" if count == 2 else "#9a8c98"
    for count in top20_aggregate["target_top20_count"]
]
ax.barh(top20_aggregate["feature"], top20_aggregate["target_top20_count"], color=colors)
ax.set_xlabel("Number of targets in permutation Top 20")
ax.set_title("Cross-target feature ranking Top 20")
ax.set_xlim(0, 3.2)
fig.tight_layout()
plt.show()

# Correlation heatmap for the aggregate Top 30.
fig, ax = plt.subplots(figsize=(14, 12))
masked_corr = np.ma.masked_invalid(correlation_matrix.to_numpy())
image = ax.imshow(masked_corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(np.arange(len(top30_features)))
ax.set_yticks(np.arange(len(top30_features)))
ax.set_xticklabels(top30_features, rotation=90, fontsize=6)
ax.set_yticklabels(top30_features, fontsize=6)
ax.set_title("Pearson correlation — aggregate Top 30 features")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
fig.tight_layout()
plt.show()

## 7. 실행 요약

### Feature validation

- 기존 104개 feature baseline에서 세 target 모두 validation 예측 신호가 확인됐습니다: pressure low R²=0.782, pressure high R²=0.804, leak R²=0.781.
- FlowCool pressure high가 이번 grouped validation에서 가장 잘 설명됐습니다. train과 validation의 error 차이는 과적합 가능성을 보여주므로 이 수치는 feature check baseline으로 해석합니다.

### Feature reduction

- 기존 feature 104개에서 JEV 검토 후보 20개를 남겼습니다.
- 후보가 많이 나온 sensor는 ROTATIONSPEED (3), IONGAUGEPRESSURE (3), ETCHSUPPRESSORCURRENT (3), FLOWCOOLPRESSURE (3)입니다.
- 상위 30개에서 ACTUALROTATIONANGLE의 peak_abs/rms (r=0.971), mean/rms (r=0.983)가 강하게 중복됐습니다. 더 높은 순위 feature를 대표 후보로 표시했으며 자동 삭제하지 않았습니다.

### Next step

후보를 검토해 JEV probabilistic gate에 전달할 compact feature state를 설계합니다. 이 RF는 feature 검증용이며, 다음 구조는 validated features → JEV probabilistic gate → similar-case retrieval → Luna Judge → human/interlock 입니다.
